#📌 Extracción

In [2]:
import pandas as pd
import requests

# 1. Usamos la URL "raw" para obtener solo los datos del JSON
url_json = "https://raw.githubusercontent.com/ingridcristh/challenge2-data-science-LATAM/main/TelecomX_Data.json"

# 2. Cargar los datos directamente en un DataFrame de Pandas
try:
    # Pandas puede leer URLs directamente si el formato es correcto
    df_telecom = pd.read_json(url_json)
    print("✅ ¡Datos cargados con éxito!")

    # 3. Mostrar las primeras filas para verificar
    display(df_telecom.head())

except Exception as e:
    print(f"❌ Hubo un error al cargar los datos: {e}")

# Para ver la información de las columnas y tipos de datos
df_telecom.info()

✅ ¡Datos cargados con éxito!


,customerID,Churn,customer,phone,internet,account
0,0002-ORFBO,No,"{'gender': 'Female', 'SeniorCitizen': 0, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'One year', 'PaperlessBilling': '..."
1,0003-MKNFE,No,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'Yes'}","{'InternetService': 'DSL', 'OnlineSecurity': '...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
2,0004-TLHLJ,Yes,"{'gender': 'Male', 'SeniorCitizen': 0, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
3,0011-IGKFF,Yes,"{'gender': 'Male', 'SeniorCitizen': 1, 'Partne...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."
4,0013-EXCHZ,Yes,"{'gender': 'Female', 'SeniorCitizen': 1, 'Part...","{'PhoneService': 'Yes', 'MultipleLines': 'No'}","{'InternetService': 'Fiber optic', 'OnlineSecu...","{'Contract': 'Month-to-month', 'PaperlessBilli..."


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7267 entries, 0 to 7266
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerID  7267 non-null   object
 1   Churn       7267 non-null   object
 2   customer    7267 non-null   object
 3   phone       7267 non-null   object
 4   internet    7267 non-null   object
 5   account     7267 non-null   object
dtypes: object(6)
memory usage: 340.8+ KB


#🔧 Transformación

In [3]:
# Aplanamos las columnas que tienen diccionarios
df_customer = pd.json_normalize(df_telecom['customer'])
df_phone = pd.json_normalize(df_telecom['phone'])
df_internet = pd.json_normalize(df_telecom['internet'])
df_account = pd.json_normalize(df_telecom['account'])

# Concatenamos todo en un solo DataFrame limpio, eliminando las columnas originales anidadas
df_limpio = pd.concat([df_telecom[['customerID', 'Churn']], df_customer, df_phone, df_internet, df_account], axis=1)

print("Columnas después de aplanar:", df_limpio.columns.tolist())

# 1. Buscar valores nulos
print("Valores nulos por columna:\n", df_limpio.isnull().sum())

# 2. Verificar tipos de datos (Charges debería ser numérico, no object)
print("\nTipos de datos actuales:\n", df_limpio.dtypes)

# 3. Buscar inconsistencias en 'TotalCharges' (a veces vienen espacios vacíos " ")
incoherentes = df_limpio[df_limpio['Charges.Total'] == " "]
print(f"\nFilas con TotalCharges vacío: {len(incoherentes)}")


# A. Convertir cargos a números (forzando errores a NaN)
df_limpio['Charges.Monthly'] = pd.to_numeric(df_limpio['Charges.Monthly'], errors='coerce')
df_limpio['Charges.Total'] = pd.to_numeric(df_limpio['Charges.Total'], errors='coerce')

# B. Llenar valores nulos en TotalCharges
# Si el cliente es nuevo (tenure=0), el total es 0. Si no, podemos llenarlo con la media o eliminarlo.
df_limpio['Charges.Total'] = df_limpio['Charges.Total'].fillna(0)

# C. Estandarizar textos (opcional pero recomendado)
# Eliminar espacios extra en nombres de categorías si existieran
df_limpio = df_limpio.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

print("✅ Correcciones aplicadas. El dataset está listo para el análisis.")
display(df_limpio.head())


# 1. Creamos la nueva columna dividiendo el cargo mensual por 30
# Usamos .round(2) para que solo tenga dos decimales (formato moneda)
df_limpio['Cuentas_Diarias'] = (df_limpio['Charges.Monthly'] / 30).round(2)

# 2. Verificamos la creación de la columna mostrando las columnas relevantes
display(df_limpio[['customerID', 'Charges.Monthly', 'Cuentas_Diarias']].head())

# 3. Análisis rápido: ¿Cuál es el gasto diario promedio de toda la base?
gasto_diario_promedio = df_limpio['Cuentas_Diarias'].mean()
print(f"\n💰 El valor diario promedio por cliente es de: ${gasto_diario_promedio:.2f}")

# Diccionario de traducción para las columnas principales
columnas_espanol = {
    'customerID': 'ID_Cliente',
    'Churn': 'Abandono',
    'gender': 'Genero',
    'SeniorCitizen': 'Adulto_Mayor',
    'Partner': 'Tiene_Pareja',
    'Dependents': 'Dependientes',
    'tenure': 'Meses_Permanencia',
    'PhoneService': 'Servicio_Telefonico',
    'MultipleLines': 'Lineas_Multiples',
    'InternetService': 'Tipo_Internet',
    'OnlineSecurity': 'Seguridad_Online',
    'OnlineBackup': 'Respaldo_Online',
    'DeviceProtection': 'Proteccion_Dispositivo',
    'TechSupport': 'Soporte_Tecnico',
    'StreamingTV': 'Streaming_TV',
    'StreamingMovies': 'Streaming_Peliculas',
    'Contract': 'Tipo_Contrato',
    'PaperlessBilling': 'Factura_Digital',
    'PaymentMethod': 'Metodo_Pago',
    'Charges.Monthly': 'Cargos_Mensuales',
    'Charges.Total': 'Cargos_Totales'
}

df_limpio.rename(columns=columnas_espanol, inplace=True)

# Creamos un mapa de reemplazo
mapa_binario = {'Yes': 1, 'No': 0, 'Female': 1, 'Male': 0}

# Aplicamos la transformación a las columnas que solo tienen esas opciones
columnas_a_binarizar = ['Abandono', 'Tiene_Pareja', 'Dependientes', 'Servicio_Telefonico', 'Factura_Digital', 'Genero']

for col in columnas_a_binarizar:
    if col in df_limpio.columns:
        df_limpio[col] = df_limpio[col].map(mapa_binario)

# Nota: Algunas columnas tienen 'No internet service', esas las dejamos como texto por ahora.

# Traducir categorías de Contrato y Método de Pago
traduccion_categorias = {
    'Month-to-month': 'Mensual',
    'One year': 'Anual',
    'Two year': 'Bianual',
    'Electronic check': 'Cheque Electrónico',
    'Mailed check': 'Cheque por Correo',
    'Bank transfer (automatic)': 'Transferencia Bancaria',
    'Credit card (automatic)': 'Tarjeta de Crédito'
}

df_limpio['Tipo_Contrato'] = df_limpio['Tipo_Contrato'].replace(traduccion_categorias)
df_limpio['Metodo_Pago'] = df_limpio['Metodo_Pago'].replace(traduccion_categorias)

print("✅ Datos estandarizados y traducidos.")
display(df_limpio[['ID_Cliente', 'Abandono', 'Tipo_Contrato', 'Cargos_Mensuales', 'Cuentas_Diarias']].head())

Columnas después de aplanar: ['customerID', 'Churn', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Charges.Monthly', 'Charges.Total']
Valores nulos por columna:
 customerID          0
Churn               0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
Charges.Monthly     0
Charges.Total       0
dtype: int64

Tipos de datos actuales:
 customerID           object
Churn                object
gender               object
SeniorCitizen         int64
Par

,customerID,Churn,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,...,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,Charges.Monthly,Charges.Total
0,0002-ORFBO,No,Female,0,Yes,Yes,9,Yes,No,DSL,...,Yes,No,Yes,Yes,No,One year,Yes,Mailed check,65.6,593.30
1,0003-MKNFE,No,Male,0,No,No,9,Yes,Yes,DSL,...,No,No,No,No,Yes,Month-to-month,No,Mailed check,59.9,542.40
2,0004-TLHLJ,Yes,Male,0,No,No,4,Yes,No,Fiber optic,...,No,Yes,No,No,No,Month-to-month,Yes,Electronic check,73.9,280.85
3,0011-IGKFF,Yes,Male,1,Yes,No,13,Yes,No,Fiber optic,...,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,98.0,1237.85
4,0013-EXCHZ,Yes,Female,1,Yes,No,3,Yes,No,Fiber optic,...,No,No,Yes,Yes,No,Month-to-month,Yes,Mailed check,83.9,267.40


,customerID,Charges.Monthly,Cuentas_Diarias
0,0002-ORFBO,65.6,2.19
1,0003-MKNFE,59.9,2.00
2,0004-TLHLJ,73.9,2.46
3,0011-IGKFF,98.0,3.27
4,0013-EXCHZ,83.9,2.80



💰 El valor diario promedio por cliente es de: $2.16
✅ Datos estandarizados y traducidos.


,ID_Cliente,Abandono,Tipo_Contrato,Cargos_Mensuales,Cuentas_Diarias
0,0002-ORFBO,0.0,Anual,65.6,2.19
1,0003-MKNFE,0.0,Mensual,59.9,2.00
2,0004-TLHLJ,1.0,Mensual,73.9,2.46
3,0011-IGKFF,1.0,Mensual,98.0,3.27
4,0013-EXCHZ,1.0,Mensual,83.9,2.80


#📊 Carga y análisis

In [4]:
# Seleccionamos las columnas clave para el análisis de negocio
columnas_analisis = ['Meses_Permanencia', 'Cargos_Mensuales', 'Cargos_Totales', 'Cuentas_Diarias']

# Generamos el resumen estadístico
resumen_stats = df_limpio[columnas_analisis].describe()

# Añadimos la Mediana manualmente (que es el percentil 50, pero para claridad la mostramos aparte)
resumen_stats.loc['median'] = df_limpio[columnas_analisis].median()

display(resumen_stats)

import plotly.express as px

# Histograma de permanencia para ver la lealtad
fig = px.histogram(df_limpio, x='Meses_Permanencia', nbins=30,
                   title='Distribución de la Permanencia de Clientes (Meses)',
                   labels={'Meses_Permanencia': 'Meses en la Compañía'},
                   color_discrete_sequence=['#636EFA'])
fig.show()

# Comparativa de promedios según si abandonaron o no
comparativa_abandono = df_limpio.groupby('Abandono')[columnas_analisis].mean()
print("📊 Promedios: Clientes que se Quedan (0) vs Clientes que se Van (1)")
display(comparativa_abandono)

# Contamos los valores de la columna Abandono
conteo_abandono = df_limpio['Abandono'].value_counts().reset_index()
conteo_abandono.columns = ['Estado', 'Total_Clientes']

# Reemplazamos los números por etiquetas para que el gráfico sea profesional
conteo_abandono['Estado'] = conteo_abandono['Estado'].replace({1: 'Se Fueron (Churn)', 0: 'Permanecen'})

print(conteo_abandono)

import plotly.express as px

fig_pie = px.pie(conteo_abandono,
                 values='Total_Clientes',
                 names='Estado',
                 title='Proporción de Abandono de Clientes (Churn)',
                 color='Estado',
                 color_discrete_map={'Permanecen':'#2ECC71', 'Se Fueron (Churn)':'#E74C3C'},
                 hole=0.4) # Lo hacemos tipo dona para que se vea más moderno

fig_pie.update_traces(textinfo='percent+label')
fig_pie.show()

fig_bar = px.bar(conteo_abandono,
                 x='Estado',
                 y='Total_Clientes',
                 text='Total_Clientes',
                 title='Cantidad Total de Clientes por Estado de Abandono',
                 color='Estado',
                 color_discrete_map={'Permanecen':'#2ECC71', 'Se Fueron (Churn)':'#E74C3C'})

fig_bar.update_traces(textposition='outside')
fig_bar.show()

import plotly.express as px

# Gráfico de Abandono por Tipo de Contrato
fig_contrato = px.histogram(df_limpio, x="Tipo_Contrato", color="Abandono",
                             barmode="group", title="Distribución de Abandono por Tipo de Contrato",
                             labels={'Abandono': 'Estado (1=Se fue, 0=Sigue)'},
                             color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_contrato.show()

# Gráfico de Abandono por Método de Pago
fig_pago = px.histogram(df_limpio, y="Metodo_Pago", color="Abandono",
                         barmode="group", title="Abandono según Método de Pago",
                         orientation='h', # Horizontal para leer mejor los nombres largos
                         color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_pago.show()

# Gráfico de Abandono por Género
fig_genero = px.histogram(df_limpio, x="Genero", color="Abandono",
                           barmode="group", title="Abandono por Género (1=Mujer, 0=Hombre)",
                           color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_genero.show()

# Gráfico de Abandono por Adulto Mayor
fig_senior = px.histogram(df_limpio, x="Adulto_Mayor", color="Abandono",
                           barmode="group", title="Abandono: Clientes Jóvenes (0) vs Adultos Mayores (1)",
                           color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_senior.show()

import plotly.express as px

# Boxplot para comparar la permanencia
fig_permanencia = px.box(df_limpio, x='Abandono', y='Meses_Permanencia',
                         color='Abandono',
                         title='Distribución de Meses de Permanencia vs Abandono',
                         labels={'Abandono': 'Estado (0=Se queda, 1=Se fue)'},
                         color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_permanencia.show()

# Histograma de densidad para Cargos Mensuales
fig_mensual = px.histogram(df_limpio, x="Cargos_Mensuales", color="Abandono",
                           marginal="box", # Añade un boxplot arriba para ver medianas
                           title="Relación entre Cargos Mensuales y Abandono",
                           color_discrete_map={0:'#2ECC71', 1:'#E74C3C'},
                           barmode="overlay") # Superpone los colores para comparar áreas
fig_mensual.show()

# Gráfico de violín para ver la densidad del gasto diario
fig_diario = px.violin(df_limpio, y="Cuentas_Diarias", x="Abandono",
                       color="Abandono", box=True, points="all",
                       title="Distribución del Gasto Diario vs Abandono",
                       color_discrete_map={0:'#2ECC71', 1:'#E74C3C'})
fig_diario.show()

,Meses_Permanencia,Cargos_Mensuales,Cargos_Totales,Cuentas_Diarias
count,7267.000000,7267.000000,7267.000000,7267.000000
mean,32.346498,64.720098,2277.182035,2.157292
std,24.571773,30.129572,2268.648587,1.004407
min,0.000000,18.250000,0.000000,0.610000
25%,9.000000,35.425000,396.200000,1.180000
50%,29.000000,70.300000,1389.200000,2.340000
75%,55.000000,89.875000,3778.525000,2.995000
max,72.000000,118.750000,8684.800000,3.960000
median,29.000000,70.300000,1389.200000,2.340000


📊 Promedios: Clientes que se Quedan (0) vs Clientes que se Van (1)


,Meses_Permanencia,Cargos_Mensuales,Cargos_Totales,Cuentas_Diarias
Abandono,,,,
0.0,37.569965,61.265124,2549.911442,2.04208
1.0,17.979133,74.441332,1531.796094,2.48145


              Estado  Total_Clientes
0         Permanecen            5174
1  Se Fueron (Churn)            1869


#📄Informe final

🔹 Introducción
El objetivo de este análisis es comprender las causas detrás de la evasión de clientes (Churn) en la empresa TelecomX. La pérdida de clientes es uno de los desafíos más críticos en el sector de telecomunicaciones, ya que adquirir un nuevo cliente es significativamente más costoso que retener a uno actual. Este informe busca identificar patrones de comportamiento y factores de riesgo para diseñar estrategias de retención efectivas.

2. 🔹 Limpieza y Tratamiento de Datos
Para garantizar la calidad de los resultados, se siguieron los siguientes pasos técnicos:

Importación: Los datos se consumieron directamente desde la API de GitHub en formato JSON.

Aplanamiento (Flattening): Se normalizaron estructuras anidadas de diccionarios en columnas individuales (datos de cliente, servicios y cuentas).

Corrección de Incoherencias: Se transformaron columnas de texto a numéricas (como Cargos_Totales) y se gestionaron valores nulos o espacios vacíos.

Estandarización: Se tradujeron las variables al español y se binarizaron columnas críticas (1 para 'Sí' / 0 para 'No') para facilitar el procesamiento matemático.

Ingeniería de Datos: Se creó la métrica Cuentas_Diarias para analizar el gasto proporcional por día.

3. 🔹 Análisis Exploratorio de Datos (EDA)
A través de la visualización, identificamos los siguientes pilares del comportamiento del cliente:

Distribución del Churn: Observamos que la tasa de abandono se concentra en un [X]% de la base total.

Perfil del Cliente: Los clientes con contratos Mes a Mes muestran una tendencia de abandono drásticamente superior a los de contratos anuales.

Factores Económicos: Existe una correlación positiva entre los Cargos_Mensuales altos y la probabilidad de fuga. Los clientes que pagan más de $[Valor] son los más propensos a irse.

Métodos de Pago: El uso de Cheque Electrónico se asocia con niveles de evasión más altos en comparación con los métodos automáticos.

4. 🔹 Conclusiones e Insights
Lealtad Temprana: La mayoría de las fugas ocurren en los primeros [X] meses, lo que indica que el periodo de adaptación al servicio es crítico.

Sensibilidad al Precio: El costo diario (Cuentas_Diarias) es un predictor más preciso de fuga que el cargo total acumulado.

Servicios de Retención: Los clientes que cuentan con servicios de Seguridad Online o Soporte Técnico tienden a permanecer más tiempo, actuando estos como "anclas" de lealtad.

5. 🔹 Recomendaciones Estratégicas
Basado en los datos, se sugieren las siguientes acciones:

Plan de Migración de Contratos: Incentivar a los clientes de planes "Mes a Mes" hacia planes anuales mediante descuentos progresivos.

Automatización de Pagos: Promover el uso de tarjeta de crédito o transferencia bancaria para reducir la fricción de los pagos manuales (Cheques).

Alertas de "Alto Riesgo": Implementar un sistema de alertas para clientes nuevos con cargos altos, ofreciendo un seguimiento proactivo de soporte técnico durante los primeros 90 días.

Paquetes de Valor: Incluir servicios de seguridad digital de forma gratuita o bonificada en los planes de mayor costo para aumentar el valor percibido y reducir la sensibilidad al precio.